# Lab 4 — Delta Time Travel, Restore & Table Cloning

**Goal:** Inspect version history, query an earlier snapshot, and create shallow/deep clones.

**Important:** Clone support and syntax depend on your Databricks runtime/catalog configuration. Use a Unity Catalog managed Delta table and run the cells in order. The clone statements may be skipped if your workspace does not support them.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.lab_db.time_travel_lab
USING DELTA
AS SELECT * FROM VALUES
  (1, 'Alpha', 10),
  (2, 'Beta', 20)
AS t(id, item, quantity);

In [0]:
%sql
-- Create another version
UPDATE workspace.lab_db.time_travel_lab
SET quantity = quantity + 5
WHERE id = 1;

INSERT INTO workspace.lab_db.time_travel_lab VALUES (3, 'Gamma', 30);

In [0]:
%sql
DESCRIBE HISTORY workspace.lab_db.time_travel_lab;

In [0]:
%sql
-- View the earliest retained version (version 0, if still available)
SELECT * FROM workspace.lab_db.time_travel_lab VERSION AS OF 0 ORDER BY id;

In [0]:
%sql
-- Compare with the current snapshot
SELECT * FROM workspace.lab_db.time_travel_lab ORDER BY id;

## Optional cloning exercises

Run each clone statement separately. If the table already exists, use a new target name or drop the old clone first.

In [0]:
%sql
CREATE TABLE workspace.lab_db.time_travel_shallow_clone
SHALLOW CLONE workspace.lab_db.time_travel_lab;

In [0]:
%sql
CREATE TABLE workspace.lab_db.time_travel_deep_clone
DEEP CLONE workspace.lab_db.time_travel_lab;

In [0]:
%sql
SELECT 'source' AS table_name, COUNT(*) AS row_count FROM workspace.lab_db.time_travel_lab
UNION ALL
SELECT 'shallow_clone', COUNT(*) FROM workspace.lab_db.time_travel_shallow_clone
UNION ALL
SELECT 'deep_clone', COUNT(*) FROM workspace.lab_db.time_travel_deep_clone;